### Load Employee Data

In [0]:
employees_df = spark.read.csv(
    "/Volumes/workspace/default/Volume/Employees_500.csv",
    header=True,
    inferSchema=True
)
display(employees_df)


employee_id,name,email,department,salary,ssn,role,country
1001,Tqodyr Ofiopol,tqodyr.ofiopol@company.com,HR,94935,541-85-3769,Manager,USA
1002,Itctc Qgnulliu,itctc.qgnulliu@company.com,Marketing,147183,612-87-7888,Engineer,India
1003,Ukhmcdp Wbafqbha,ukhmcdp.wbafqbha@company.com,Finance,110631,180-50-8301,Manager,UK
1004,Orvnv Jbxfinmx,orvnv.jbxfinmx@company.com,Finance,117257,398-85-1026,Auditor,UK
1005,Pwini Maijisz,pwini.maijisz@company.com,Sales,71706,520-87-7233,Auditor,Australia
1006,Jepewjn Exuuboi,jepewjn.exuuboi@company.com,HR,68434,393-64-8186,Auditor,Canada
1007,Othqh Ckxvnfeje,othqh.ckxvnfeje@company.com,IT,139561,280-16-1354,Admin,Australia
1008,Vjnqsmk Sfezgz,vjnqsmk.sfezgz@company.com,Operations,160510,480-38-7466,Manager,Canada
1009,Fhdodok Szhmgacd,fhdodok.szhmgacd@company.com,Sales,164376,973-97-2745,Engineer,Canada
1010,Eflgw Oawhmt,eflgw.oawhmt@company.com,Finance,111275,883-93-2466,Analyst,India


### Identify Sensitive Data

| Column | Type |
| --- | --- |
| salary | Confidential |
| ssn | PII |
| email | PII |

In [0]:
analyst_view = employees_df.select(
    "employee_id",
    "name",
    "department",
    "email",
    "role"
)

display(analyst_view)


employee_id,name,department,email,role
1001,Tqodyr Ofiopol,HR,tqodyr.ofiopol@company.com,Manager
1002,Itctc Qgnulliu,Marketing,itctc.qgnulliu@company.com,Engineer
1003,Ukhmcdp Wbafqbha,Finance,ukhmcdp.wbafqbha@company.com,Manager
1004,Orvnv Jbxfinmx,Finance,orvnv.jbxfinmx@company.com,Auditor
1005,Pwini Maijisz,Sales,pwini.maijisz@company.com,Auditor
1006,Jepewjn Exuuboi,HR,jepewjn.exuuboi@company.com,Auditor
1007,Othqh Ckxvnfeje,IT,othqh.ckxvnfeje@company.com,Admin
1008,Vjnqsmk Sfezgz,Operations,vjnqsmk.sfezgz@company.com,Manager
1009,Fhdodok Szhmgacd,Sales,fhdodok.szhmgacd@company.com,Engineer
1010,Eflgw Oawhmt,Finance,eflgw.oawhmt@company.com,Analyst


### Create Secure View for Analysts
#### Hide:
- salary 
- ssn 


In [0]:
analyst_view = employees_df.select(
    "employee_id",
    "name",
    "department",
    "email",
    "role"
)

display(analyst_view)


employee_id,name,department,email,role
1001,Tqodyr Ofiopol,HR,tqodyr.ofiopol@company.com,Manager
1002,Itctc Qgnulliu,Marketing,itctc.qgnulliu@company.com,Engineer
1003,Ukhmcdp Wbafqbha,Finance,ukhmcdp.wbafqbha@company.com,Manager
1004,Orvnv Jbxfinmx,Finance,orvnv.jbxfinmx@company.com,Auditor
1005,Pwini Maijisz,Sales,pwini.maijisz@company.com,Auditor
1006,Jepewjn Exuuboi,HR,jepewjn.exuuboi@company.com,Auditor
1007,Othqh Ckxvnfeje,IT,othqh.ckxvnfeje@company.com,Admin
1008,Vjnqsmk Sfezgz,Operations,vjnqsmk.sfezgz@company.com,Manager
1009,Fhdodok Szhmgacd,Sales,fhdodok.szhmgacd@company.com,Engineer
1010,Eflgw Oawhmt,Finance,eflgw.oawhmt@company.com,Analyst


### Save Restricted Table

In [0]:
analyst_view.write.mode("overwrite").saveAsTable(
    "employees_analyst_view"
)


## Admin Full Access View
### Save Admin Dataset

In [0]:
employees_df.write.mode("overwrite").saveAsTable(
    "employees_admin_view"
)



### Concept

| Role | Access |
| :--- | :--- |
| Admin | Full dataset |
| Analyst | Filtered dataset |

## Simulate Authentication System

### Define Users


In [0]:
users = {
    "admin1": "Admin",
    "analyst1": "Analyst"
}


### Simulate Login

In [0]:
username = "analyst1"

role = users.get(username)

print("Logged in as:", role)


Logged in as: Analyst


### Access Logic

In [0]:
if role == "Admin":
    display(employees_df)
else:
    display(analyst_view)


employee_id,name,department,email,role
1001,Tqodyr Ofiopol,HR,tqodyr.ofiopol@company.com,Manager
1002,Itctc Qgnulliu,Marketing,itctc.qgnulliu@company.com,Engineer
1003,Ukhmcdp Wbafqbha,Finance,ukhmcdp.wbafqbha@company.com,Manager
1004,Orvnv Jbxfinmx,Finance,orvnv.jbxfinmx@company.com,Auditor
1005,Pwini Maijisz,Sales,pwini.maijisz@company.com,Auditor
1006,Jepewjn Exuuboi,HR,jepewjn.exuuboi@company.com,Auditor
1007,Othqh Ckxvnfeje,IT,othqh.ckxvnfeje@company.com,Admin
1008,Vjnqsmk Sfezgz,Operations,vjnqsmk.sfezgz@company.com,Manager
1009,Fhdodok Szhmgacd,Sales,fhdodok.szhmgacd@company.com,Engineer
1010,Eflgw Oawhmt,Finance,eflgw.oawhmt@company.com,Analyst


## Data Masking
### Mask Email

In [0]:
from pyspark.sql.functions import regexp_replace

masked_df = analyst_view.withColumn(
    "email",
    regexp_replace("email", "(^.).*(@.*$)", "\\1***\\2")
)

display(masked_df)


employee_id,name,department,email,role
1001,Tqodyr Ofiopol,HR,1***2,Manager
1002,Itctc Qgnulliu,Marketing,1***2,Engineer
1003,Ukhmcdp Wbafqbha,Finance,1***2,Manager
1004,Orvnv Jbxfinmx,Finance,1***2,Auditor
1005,Pwini Maijisz,Sales,1***2,Auditor
1006,Jepewjn Exuuboi,HR,1***2,Auditor
1007,Othqh Ckxvnfeje,IT,1***2,Admin
1008,Vjnqsmk Sfezgz,Operations,1***2,Manager
1009,Fhdodok Szhmgacd,Sales,1***2,Engineer
1010,Eflgw Oawhmt,Finance,1***2,Analyst
